## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# CNN applied to videos - part 1

To analyze movement, we must analyze videos, because videos capture movement.

A video is a sequence of images (**frames**), recorded one shortly after the other.

<div style="text-align:center">
<img src="images/imagesequence.png" alt="plot" width="800" />
</div>

So we can analyze a video by analyzing the sequence of images that make the video.

All the frames of a video have the same size. The number of frames recorded each second is the **frame rate**, in frames per second (FPS). Common values are 30 FPS (33 ms between frames) and 60 FPS (16.7 ms between frames).

The frame rate determines how much detail of the movement we can see: a low frame rate can miss fast movements, and a high frame rate gives very large files with frames that are very similar.

In notebook 07.01, we analyzed the TUG test from frames saved as image files. This is slow (each frame is saved and then read again), and we did not know the time of each frame.

A better solution is a library that reads the video directly, one frame at a time, when we ask for it. One of the most used libraries is **OpenCV** (Open Computer Vision). OpenCV has many computer vision tools; here we use it only to read video frames. OpenCV is already installed in Colab. On your own computer: `pip install opencv-python`. In Colab, install ultralytics with the next cell.

In [ ]:
!pip install -q ultralytics

### Learning Objective

What you will learn in this notebook:
- What a video is from a data perspective
- How to read a video frame by frame with OpenCV
- How a video frame is represented in Python
- Why videos can be processed one image at a time

In [ ]:
import cv2                           # OpenCV: image processing and computer vision
import matplotlib.pyplot as plt
from ultralytics import YOLO
import numpy as np
import pandas as pd

In [ ]:
# open the TUG video with OpenCV
video = cv2.VideoCapture("data/TUG.mp4")

`video` is now an object that contains information about the video

In [ ]:
print("Video loaded successfully:", video.isOpened())
print("Total number of frames in the video:", int(video.get(cv2.CAP_PROP_FRAME_COUNT)))
print("Frames per second (FPS) of the video:", video.get(cv2.CAP_PROP_FPS))
print("Width of the video frames:", int(video.get(cv2.CAP_PROP_FRAME_WIDTH)))
print("Height of the video frames:", int(video.get(cv2.CAP_PROP_FRAME_HEIGHT)))
print(f"Duration of the video (seconds): {video.get(cv2.CAP_PROP_FRAME_COUNT) / video.get(cv2.CAP_PROP_FPS)} s")

Note: this information is not always correct. Some programs do not store the video information correctly.

You can also access the individual frames

In [ ]:
valid, frame = video.read()     # read the next frame (here: the first frame)
print("Frame read successfully:", valid)
print("Frame dimensions:", frame.shape)
print("Frame time stamp (seconds):", video.get(cv2.CAP_PROP_POS_MSEC) / 1000)

You can plot the frame:

In [ ]:
plt.imshow(frame)
plt.show()

The colors are wrong. OpenCV stores the color channels in the order **BGR** (blue, green, red), but Matplotlib expects **RGB** (red, green, blue). This is a historical choice of OpenCV. We convert the frame before we plot it:

In [ ]:
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))   # BGR -> RGB, only for plotting
plt.show()

To read the next frame, call `read()` again:

In [ ]:
valid, frame = video.read()     # the second frame
print("Frame time stamp (seconds):", video.get(cv2.CAP_PROP_POS_MSEC) / 1000)
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.show()

Each call to `read()` gives the next frame. To go to a specific frame (or back to the start), use `set`:

```Python
video.set(cv2.CAP_PROP_POS_FRAMES, 0)     # go back to the first frame
```

When you are done, close the video with `video.release()`.

In [ ]:
video.set(cv2.CAP_PROP_POS_FRAMES, 100)   # go to frame 100
valid, frame = video.read()               # read frame 100
print("Frame time stamp (seconds):", video.get(cv2.CAP_PROP_POS_MSEC) / 1000)
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.show()
video.release()                           # close the video

To read all the frames, we use a `while` loop that stops when `read()` fails (the end of the video):
- read a frame
- process it
- stop at the end of the video

We print only every 100th frame, to keep the output short.

In [ ]:
video = cv2.VideoCapture("data/TUG.mp4")
n = 0
while True:
    valid, frame = video.read()
    if not valid:
        break
    if n % 100 == 0:
        print(f"Frame {n}: time = {video.get(cv2.CAP_PROP_POS_MSEC) / 1000:.2f} s")
    n += 1
video.release()
print("Frames read:", n)

## Key idea
We rarely process a whole video at once. Instead, we read one frame at a time.

### Why process frame by frame?
A frame-by-frame approach lets us:
- inspect the video over time
- apply image-processing steps to each frame
- run a CNN on each frame
- extract motion information from changes across frames

This is the bridge between image analysis and video analysis.

## Activity

Read the video from frame 50 to frame 150 (use `set`, then a loop that stops after 100 frames), and print the time of the first and the last frame.

In [ ]:
# Write your code here

## Detect the person and the chair in the video

We repeat the analysis of 07.01, but now we read the frames from the video and we know the **time** of each frame.

Note: YOLO expects frames from OpenCV in the **BGR** order, so we give the frame directly to the model (no conversion). We convert to RGB only for plotting.

We store the frame number **before** reading the frame (after `read()`, OpenCV points to the next frame).

In [ ]:
model = YOLO("../models/yolo26s.pt")
name_to_id = {name: idx for idx, name in model.names.items()}
target_ids = [name_to_id['person'], name_to_id['chair']]

def detect_in_video(video_path, use_tracking=False):
    video = cv2.VideoCapture(video_path)
    positions = []
    while True:
        frame_number = int(video.get(cv2.CAP_PROP_POS_FRAMES))
        valid, frame = video.read()
        if not valid:
            break
        if use_tracking:
            r = model.track(frame, classes=target_ids, persist=True, verbose=False)[0]
        else:
            r = model(frame, classes=target_ids, verbose=False)[0]
        row = {"frame": frame_number, "time": video.get(cv2.CAP_PROP_POS_MSEC) / 1000,
               "person_x": np.nan, "person_y": np.nan, "chair_x": np.nan, "chair_y": np.nan}
        if r.boxes is not None and len(r.boxes) > 0:
            cls = r.boxes.cls.cpu().numpy().astype(int)
            conf = r.boxes.conf.cpu().numpy()
            xyxy = r.boxes.xyxy.cpu().numpy()
            for label in ["person", "chair"]:
                rows = np.where(cls == name_to_id[label])[0]
                if rows.size > 0:
                    best = rows[np.argmax(conf[rows])]
                    x1, y1, x2, y2 = xyxy[best]
                    row[f"{label}_x"], row[f"{label}_y"] = (x1 + x2) / 2, (y1 + y2) / 2
        positions.append(row)
    video.release()
    return pd.DataFrame(positions)

def plot_positions(df, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(df["time"], df["person_x"]); ax[0].set_title(f"{title}: horizontal position")
    ax[1].plot(df["time"], df["person_y"]); ax[1].set_title(f"{title}: vertical position")
    ax[1].invert_yaxis()      # image coordinates: y increases downwards
    for a in ax:
        a.set_xlabel("Time (s)"); a.set_ylabel("Position (pixels)")
    plt.show()

df = detect_in_video("data/TUG.mp4")
plot_positions(df, "Detection")

## Tracking

Many models can also **track** objects in a video. Tracking links the detections from one frame to the next and gives each object an **ID** that stays the same over time. This is useful when there are several people: we can follow the same person in the whole video.

Tracking does not make the box of each frame more accurate. It helps to keep the identity of each object, and to fill short gaps where a detection is missing.

In [ ]:
df_track = detect_in_video("data/TUG.mp4", use_tracking=True)
plot_positions(df_track, "Tracking")

Look for sudden jumps in the signals: they are usually errors of the detection. Let us find the frame with the largest jump in the vertical position, and look at it.

In [ ]:
jump = df["person_y"].diff().abs()
f = int(df.loc[jump.idxmax(), "frame"])
print(f"Largest jump: frame {f}, time {df.loc[jump.idxmax(), 'time']:.2f} s, jump = {jump.max():.1f} pixels")
video = cv2.VideoCapture("data/TUG.mp4")
video.set(cv2.CAP_PROP_POS_FRAMES, f)
valid, frame = video.read()
video.release()
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f'Frame {f}')
plt.show()

The jump is small here (a few pixels): the box changes size when the arms or legs move, so its center moves a little. Larger errors happen when the person is hidden, when another object is detected as a person, or with motion blur.

Note also that the video information said 457 frames, but the video has only 319 frames (12.8 s). Always check the information of a video.

**Questions**
- Use the plots to estimate the duration of the TUG test, in seconds.
- What can cause the errors in the signal (occlusion, the person turning, motion blur)?